In [14]:
import duckdb
import polars as pl
import pendulum
from pathlib import Path

data_dir = Path("../data/lastfm/listening")
db = duckdb.read_json(data_dir/"*.jsonl")
df = db.pl()

df = df.with_columns(
    pl.from_epoch(
        pl.col("date_played_unix"), time_unit="s"
    ).alias("track_played_utc")
)
print("Raw data:", df.shape)

df = df.unique("date_played_unix") # Not so exact with which duplicate gets removed
print("Duplicate timestamps removed:", df.unique("date_played_unix").shape)

Raw data: (110513, 8)
Duplicate timestamps removed: (108066, 8)


In [15]:
( 
    df
    .sort(pl.col("date_played_unix"), descending=False) 
    .filter(pl.col("track_played_utc").dt.year() == 2025)
    .with_columns(
        played_delta_s = pl.col("date_played_unix").diff().fill_null(0)
    ).filter(pl.col("date_played_unix") <= 1741521825)
).sort(pl.col("date_played_unix"), descending=True).head(2)

artist_name,artist_mbid,album_name,album_mbid,track_name,track_mbid,date_played_unix,track_played_utc,played_delta_s
str,str,str,str,str,str,i64,datetime[μs],i64
"""Dean Blunt""","""e8bd5b47-e8b4-4671-a9f6-590a92…","""Black Metal 2""","""128051d5-20f6-4abe-a30e-661cf5…","""VIGIL""","""99bb312d-1521-4603-8b60-a9580f…",1741521825,2025-03-09 12:03:45,149
"""Dean Blunt""","""""","""BLACK METAL 2""","""""","""VIGIL""","""""",1741521676,2025-03-09 12:01:16,283


Loading data from Spotify exported streaming history

In [16]:
df.columns

['artist_name',
 'artist_mbid',
 'album_name',
 'album_mbid',
 'track_name',
 'track_mbid',
 'date_played_unix',
 'track_played_utc']

In [17]:
from pathlib import Path
spotify_dir = Path("../data/spotify")
spotify_db = duckdb.read_json(spotify_dir/"*.json")
spotify_df = spotify_db.pl()
spotify_df.head()

ts,platform,ms_played,conn_country,ip_addr,master_metadata_track_name,master_metadata_album_artist_name,master_metadata_album_album_name,spotify_track_uri,episode_name,episode_show_name,spotify_episode_uri,audiobook_title,audiobook_uri,audiobook_chapter_uri,audiobook_chapter_title,reason_start,reason_end,shuffle,skipped,offline,offline_timestamp,incognito_mode
datetime[μs],str,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,bool,bool,bool,i64,bool
2024-11-17 15:51:57,"""osx""",150092,"""NO""","""193.156.58.159""","""Movement 9""","""Floating Points""","""Promises""","""spotify:track:2PGBGqkjFsAHErsw…",null,null,null,null,null,null,null,"""trackdone""","""endplay""",false,true,false,1731858566,false
2024-11-17 15:53:42,"""osx""",104506,"""NO""","""193.156.58.159""","""Michelle""","""Yusef Lateef""","""Suite 16""","""spotify:track:3YmxwafE7wiBGOM7…",null,null,null,null,null,null,null,"""trackdone""","""trackdone""",false,false,false,1731858718,false
2024-11-17 15:56:02,"""osx""",139586,"""NO""","""193.156.58.159""","""Whisky Story Time""","""Alabaster DePlume""","""To Cy & Lee: Instrumentals Vol…","""spotify:track:1ZBxqoho42uvFAJB…",null,null,null,null,null,null,null,"""trackdone""","""trackdone""",false,false,false,1731858822,false
2024-11-17 15:58:39,"""osx""",155406,"""NO""","""193.156.58.159""","""Electric Counterpoint: II. Slo…","""Steve Reich""","""Steve Reich: Electric Counterp…","""spotify:track:6Xcnbv35WPrlYPJ0…",null,null,null,null,null,null,null,"""trackdone""","""endplay""",false,true,false,1731858962,false
2024-11-17 15:59:31,"""osx""",50677,"""NO""","""193.156.58.159""","""Number 2 (feat. Future & 21 Sa…","""KSI""","""All Over The Place""","""spotify:track:5RnUNH9TYaNN87ul…",null,null,null,null,null,null,null,"""clickrow""","""endplay""",false,true,false,1731859119,false


In [18]:
new_col_names = {
    'ts': "track_played_utc",
    'master_metadata_track_name': "track_name",
    'master_metadata_album_artist_name': "artist_name",
    'master_metadata_album_album_name': "album_name",
    'offline_timestamp': "date_played_unix",
}
spotify_df = spotify_df.select(
    [
        'ts',
        'ms_played',
        'master_metadata_track_name',
        'master_metadata_album_artist_name',
        'master_metadata_album_album_name',
        'offline_timestamp',
        # 'skipped',
    ]
).rename(new_col_names)
spotify_df

track_played_utc,ms_played,track_name,artist_name,album_name,date_played_unix
datetime[μs],i64,str,str,str,i64
2024-11-17 15:51:57,150092,"""Movement 9""","""Floating Points""","""Promises""",1731858566
2024-11-17 15:53:42,104506,"""Michelle""","""Yusef Lateef""","""Suite 16""",1731858718
2024-11-17 15:56:02,139586,"""Whisky Story Time""","""Alabaster DePlume""","""To Cy & Lee: Instrumentals Vol…",1731858822
2024-11-17 15:58:39,155406,"""Electric Counterpoint: II. Slo…","""Steve Reich""","""Steve Reich: Electric Counterp…",1731858962
2024-11-17 15:59:31,50677,"""Number 2 (feat. Future & 21 Sa…","""KSI""","""All Over The Place""",1731859119
…,…,…,…,…,…
2025-05-24 18:03:49,15519,"""Frail""","""Crystal Castles""","""Amnesty (I)""",1748095673
2025-05-24 18:12:41,524110,null,null,null,1748109836
2025-05-24 18:12:51,10430,"""Do you wanna - Fine + Molina E…","""Astrid Sonne""","""Great Doubt EDITS""",1748110361


In [19]:
spotify_df.null_count()

track_played_utc,ms_played,track_name,artist_name,album_name,date_played_unix
u32,u32,u32,u32,u32,u32
0,0,101,101,101,0


In [20]:
spotify_df.filter(pl.col("track_name").is_null())

track_played_utc,ms_played,track_name,artist_name,album_name,date_played_unix
datetime[μs],i64,str,str,str,i64
2024-11-26 05:49:01,170,null,null,null,1732600140
2025-01-22 20:09:07,158476,null,null,null,1737574683
2025-01-24 16:43:07,9287,null,null,null,1737736977
2025-01-29 12:13:16,800,null,null,null,1738152791
2025-01-29 12:13:28,12390,null,null,null,1738152793
…,…,…,…,…,…
2025-05-24 13:20:04,10770,null,null,null,1748092791
2025-05-24 13:45:06,1499502,null,null,null,1748092805
2025-05-24 13:45:16,948,null,null,null,1748094306


Last.fm scrobble rules:
1. Track must be longer than than 30 seconds 
2. and must be played for over half of its duration or for 4 minutes (whichever occurs first)

No way to check song length unless I get the track data for every song here.

For the sake of simplicity, I'll employ Spotify's definition of a "listen", listening to a track for at least 30 seconds, to narrow down valid data entries.

In [21]:
# New scrobble rule
start_of_gap = 1738221713
end_of_gap = 1741521825
missing_data = (
    spotify_df
    .filter(
        pl.col("date_played_unix") > start_of_gap,
        pl.col("date_played_unix") < end_of_gap
    )
    .filter(
        pl.col("ms_played") >= 30000
    )
).sort(pl.col("date_played_unix"), descending=False)
missing_data

track_played_utc,ms_played,track_name,artist_name,album_name,date_played_unix
datetime[μs],i64,str,str,str,i64
2025-01-30 07:25:54,107706,"""Run It""","""clipping.""","""Run It""",1738221845
2025-01-30 07:28:24,123750,"""Seconds""","""Alaskan Tapes""","""Seconds / Hólar""",1738221981
2025-01-30 07:30:36,126799,"""Suite No. 15 in D Minor for Ha…","""George Frideric Handel""","""Handel: Suites For Keyboard""",1738222104
2025-01-30 07:32:11,94946,"""Hinoki Wood""","""Gia Margaret""","""Romantic Piano""",1738222236
2025-01-30 07:33:33,80890,"""Staten Island Ferry""","""Elijah Fox""","""City in the Sky (Piano Works)""",1738222331
…,…,…,…,…,…
2025-03-09 11:48:15,194180,"""Aquamarine / Arcamarine""","""Addison Rae""","""Aquamarine / Arcamarine""",1741520700
2025-03-09 11:52:44,268016,"""Motivation""","""Clams Casino""","""Instrumentals""",1741520895
2025-03-09 11:56:33,229223,"""Mystery or Misery?""","""Vegyn""","""Mystery or Misery?""",1741521164


In [22]:
data_out = missing_data.with_columns(
    artist_mbid = pl.lit(""),
    track_mbid = pl.lit(""),
    album_mbid = pl.lit(""),
)
data_out = data_out.select(
    [
        'artist_name',
        'artist_mbid',
        'album_name',
        'album_mbid',
        'track_name',
        'track_mbid',
        'date_played_unix',
        # 'track_played_utc'
    ]
)

In [23]:
spotify_dir = Path("../data/lastfm/listening")
data_out.write_ndjson(spotify_dir/"missing_data_jan_to_march.jsonl")

Checking artists between the two tables

In [24]:
spotify_artists = spotify_df.select("artist_name").unique()
spotify_artists.filter(pl.col("artist_name") == "Smerz")

artist_name
str
"""Smerz"""


In [25]:
import re

q = "fart ing ea"
re.findall(r"(^|\s)[a-z]", q)

['', ' ', ' ']

In [26]:
re.sub(r"(^|\s)[a-z]", lambda match: match.group().upper(),q)

'Fart Ing Ea'